# Stage 10 — baseline fraud models

Compare Logistic Regression, Random Forest, and HistGradientBoosting on static SIM features and causal SIM-hour history features. Splits are grouped by SIM so the same subscriber cannot appear in both training and test data. Results on hourly rows are row-level; SIM-level aggregation is reported separately.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score)

BASE=Path('../outputs/model_ready/all_naive_12pct_200sims')
OUT=Path('../outputs/models/baseline_comparison')
OUT.mkdir(parents=True,exist_ok=True)
static=pd.read_csv(BASE/'static_sim_features.csv',dtype={'sim':'string'})
history=pd.read_csv(BASE/'history_aware_sim_hour_features.csv',dtype={'sim':'string'},parse_dates=['hour_ts'])
print('static rows:',len(static),'history rows:',len(history))

In [ ]:
models={
 'Logistic Regression':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=2000,random_state=42)),
 'Random Forest':make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(n_estimators=400,min_samples_leaf=2,class_weight='balanced_subsample',n_jobs=-1,random_state=42)),
 'HistGradientBoosting':make_pipeline(SimpleImputer(strategy='median'),HistGradientBoostingClassifier(max_iter=200,learning_rate=.08,max_leaf_nodes=15,l2_regularization=1.0,random_state=42))
}

def prepare(frame, target, group):
    excluded={target,group,'hour_ts','known_fraud','static_fraud','record_id','event_key','src_number','dst_number','src_imei','dst_imei'}
    columns=[c for c in frame.select_dtypes(include='number').columns if c not in excluded]
    X=frame[columns].replace([np.inf,-np.inf],np.nan)
    y=frame[target].astype(int)
    groups=frame[group].astype(str)
    splitter=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42)
    train_idx,test_idx=next(splitter.split(X,y,groups))
    return X,y,groups,columns,train_idx,test_idx

def evaluate(frame, name, target, group):
    X,y,groups,columns,tr,te=prepare(frame,target,group)
    print(name,'class counts:',y.value_counts().to_dict(),'features:',len(columns),'test SIMs:',groups.iloc[te].nunique())
    results=[]; fitted={}
    for model_name,model in models.items():
        model.fit(X.iloc[tr],y.iloc[tr]); pred=model.predict(X.iloc[te]); proba=model.predict_proba(X.iloc[te])[:,1]
        row={'dataset':name,'model':model_name,'precision':precision_score(y.iloc[te],pred,zero_division=0),'recall':recall_score(y.iloc[te],pred,zero_division=0),'f1':f1_score(y.iloc[te],pred,zero_division=0),'pr_auc':average_precision_score(y.iloc[te],proba),'roc_auc':roc_auc_score(y.iloc[te],proba),'test_rows':len(te),'test_sims':groups.iloc[te].nunique()}
        results.append(row); fitted[model_name]=(model,X.iloc[te],y.iloc[te],groups.iloc[te],pred,proba)
    return results,fitted,columns

In [ ]:
static_results,static_fitted,static_columns=evaluate(static,'static_SIM','static_fraud','sim')
# Static table includes a grouping key and ground truth; group by SIM, target is static_fraud.
history_results,history_fitted,history_columns=evaluate(history,'history_SIM_hour','fraud','sim')
results=pd.DataFrame(static_results+history_results)
results.to_csv(OUT/'baseline_metrics.csv',index=False)
results.sort_values(['dataset','pr_auc'],ascending=[True,False])

In [ ]:
# SIM-level ranking for the history model: maximum observed hourly probability per held-out SIM.
rank_rows=[]
for model_name,(model,Xt,yt,gt,pred,proba) in history_fitted.items():
    ranked=pd.DataFrame({'sim':gt.to_numpy(),'label':yt.to_numpy(),'probability':proba}).groupby('sim').agg(label=('label','max'),probability=('probability','max')).sort_values('probability',ascending=False).reset_index()
    for k in [10,25,50,100]:
        n=min(k,len(ranked)); top=ranked.head(n)
        rank_rows.append({'model':model_name,'top_k':n,'fraud_found':int(top.label.sum()),'precision_at_k':float(top.label.mean()),'recall_at_k':float(top.label.sum()/ranked.label.sum()) if ranked.label.sum() else np.nan})
rank_metrics=pd.DataFrame(rank_rows); rank_metrics.to_csv(OUT/'history_top_k_metrics.csv',index=False)
rank_metrics

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
for dataset,group in results.groupby('dataset'):
    ax.plot(group.model,group.pr_auc,marker='o',label=dataset)
ax.set_ylabel('PR-AUC'); ax.set_xlabel('Model'); ax.set_title('Baseline fraud model comparison'); ax.tick_params(axis='x',rotation=15); ax.legend(); fig.tight_layout(); fig.savefig(OUT/'baseline_pr_auc.png',dpi=160); plt.show()
print('Metrics saved to',OUT)

## Evaluation notes

The split is subscriber-grouped. For hourly data, reported classification metrics are calculated over test SIM-hour rows, while top-k metrics aggregate each test SIM by its maximum hourly probability. The static table represents full-day classification and cannot be interpreted as early detection. HistGradientBoosting does not expose class weights in this scikit-learn estimator, so class imbalance is handled for the other baselines only; compare its results with that limitation in mind. Threshold-dependent metrics use the default 0.5 threshold and should be tuned on a separate validation split before final reporting.